# 4. Inference (realizando la predicción)

In [14]:
import sagemaker
import numpy as np
import pandas as pd
import pprint

from sagemaker.amazon.common import numpy_to_record_serializer, record_deserializer
from sagemaker.predictor import RealTimePredictor

sess = sagemaker.Session()

In [ ]:
## I can manipulate data from dataset if I want to

###### read data dynamically
#slice=crimeArray[:,1:5]
#slice.shape
#slice
#s=slice[2:3]

In [15]:
ENDPOINT = "crime-prediction-bogota-summit"

cluster = pd.read_csv("cluster_location-co.csv")
crime = pd.read_csv('crime-colombia.csv', header=0)
xref = pd.DataFrame(crime['City'])

##### send data about the crime which I want to predict (static value)
s = np.array([[4, 8.1, 294, 80]], np.float32)

predictor = RealTimePredictor(endpoint=ENDPOINT, 
                              sagemaker_session=sess, serializer=numpy_to_record_serializer(),
                              deserializer=record_deserializer())

result = predictor.predict(s)

In [16]:
clusters = [r.label['closest_cluster'].float32_tensor.values[0] for r in result]
i = 0
    
for r in result:
    out = {
      "City" : xref['City'].iloc[i],
      "closest_cluster" :  r.label['closest_cluster'].float32_tensor.values[0],
      "crimeCluster" : crime['crimeCluster'].iloc[i],  
      "Kidnapping" :  crime['Kidnapping'].iloc[i],
      "Latitude"   :  cluster['lat'][r.label['closest_cluster'].float32_tensor.values[0]],
      "Longitude"  :  cluster['long'][r.label['closest_cluster'].float32_tensor.values[0]]
    }    
    pprint.pprint(out)
    i = i + 1

{'City': 'Bogota',
 'Kidnapping': 21.2,
 'Latitude': 4.598915389484102,
 'Longitude': -74.07380461692809,
 'closest_cluster': 6.0,
 'crimeCluster': 4}


### This is the pip command to install folium library. This environment was customized to install
### this library in Sagamaker lifecycle configuration
### This library was installed only in python 2 configuration

#!pip install git+https://github.com/psychemedia/ipython_magic_folium.git

In [17]:
#Load folium_magic. Python Library for plot maps
%load_ext folium_magic

# Set marker
marker = [[4.600851053970031,-74.07008171081543,'Operacion de equipo mas cercano!'],[
    cluster['lat'][r.label['closest_cluster'].float32_tensor.values[0]],
    cluster['long'][r.label['closest_cluster'].float32_tensor.values[0]],'Proxima area para un crimen!']]

%folium_map -l 4.60089383102632,-74.07192707061768 -z 17 -M marker -g location-bogota.json

The folium_magic extension is already loaded. To reload it, use:
  %reload_ext folium_magic
